# Probabilistic clustering (GMM and EM): lab notebook

**Machine Learning (5CS4-05), Unsupervised learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np
from scipy.stats import norm
from sklearn.mixture import GaussianMixture

### 1. One E-step and one M-step by hand (1-D, two components)

In [ ]:
# 1. One E-step and one M-step by hand (1-D, two components)
x = np.array([1.0, 2.0, 3.0, 8.0, 9.0, 10.0])
mu, sd, pi = np.array([4.0, 7.0]), np.array([2.0, 2.0]), np.array([0.5, 0.5])
dens = pi * norm.pdf(x[:, None], mu, sd)          # pi_k * N(x | mu_k, sd_k)
gamma = dens / dens.sum(axis=1, keepdims=True)    # E-step: responsibilities
print("responsibilities of component 1:", np.round(gamma[:, 0], 3))
Nk = gamma.sum(axis=0)
print("N_k:", np.round(Nk, 2), "new means:", np.round((gamma * x[:, None]).sum(axis=0) / Nk, 3), "new pi:", np.round(Nk / len(x), 3))

### 2. Full EM loop in 1-D

In [ ]:
# 2. Full EM loop in 1-D
for it in range(10):
    dens = pi * norm.pdf(x[:, None], mu, sd)
    ll = np.log(dens.sum(axis=1)).sum()
    gamma = dens / dens.sum(axis=1, keepdims=True)
    Nk = gamma.sum(axis=0)
    mu = (gamma * x[:, None]).sum(axis=0) / Nk
    sd = np.sqrt((gamma * (x[:, None] - mu) ** 2).sum(axis=0) / Nk)
    pi = Nk / len(x)
    if it in (0, 1, 4, 9):
        print(f"iter {it + 1:2d}  log-likelihood = {ll:8.3f}")
print("final means:", np.round(mu, 2), "stds:", np.round(sd, 2))

### 3. scikit-learn on 2-D data, choose K with BIC

In [ ]:
# 3. scikit-learn on 2-D data, choose K with BIC
rng = np.random.default_rng(3)
X = np.vstack([rng.normal([0, 0], 0.6, (100, 2)), rng.normal([4, 3], 0.8, (100, 2)), rng.normal([0, 5], 0.5, (100, 2))])
for K in range(1, 6):
    g = GaussianMixture(K, n_init=3, random_state=0).fit(X)
    print(f"K = {K}  BIC = {g.bic(X):8.1f}")
best = GaussianMixture(3, n_init=3, random_state=0).fit(X)
print("weights:", np.round(np.sort(best.weights_), 2))
print("soft assignment of (2, 1.5):", np.round(best.predict_proba([[2.0, 1.5]]), 3))

## Expected output

Running all cells above should print:

```
responsibilities of component 1: [0.967 0.932 0.867 0.133 0.068 0.033]
N_k: [3. 3.] new means: [2.478 8.522] new pi: [0.5 0.5]
iter  1  log-likelihood =  -16.839
iter  2  log-likelihood =  -14.331
iter  5  log-likelihood =  -11.456
iter 10  log-likelihood =  -11.456
final means: [2. 9.] stds: [0.82 0.82]
K = 1  BIC =   2613.6
K = 2  BIC =   2089.3
K = 3  BIC =   1868.7
K = 4  BIC =   1894.8
K = 5  BIC =   1923.5
weights: [0.33 0.33 0.33]
soft assignment of (2, 1.5): [[0.918 0.082 0.   ]]
```

## Practice exercises

Try these in the cells below.

1. Fit GaussianMixture with K = 1 to 6 and print the BIC for each. Which K is best?
2. Change covariance_type to "spherical" and compare the log-likelihood with "full".
3. Print predict_proba for a point between two clusters. How uncertain is it?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
